# 🔄 Evaluator-Optimizer으로 자율 개선 만들기

LangGraph 공식 문서의 **Evaluator-Optimizer 패턴**을 구현합니다. LLM이 자신의 출력을 평가하고 피드백을 반영하여 반복적으로 개선합니다.

> 📢 **Evaluator-Optimizer 패턴**
> - **Generator**: 콘텐츠 생성
> - **Evaluator**: 품질 평가 및 피드백
> - **Loop**: 조건 충족까지 반복 개선

### 🎯 학습 목표
1. `with_structured_output`으로 평가 스키마 정의
2. 조건부 엣지로 반복 루프 구현
3. 피드백 기반 자율 개선 시스템

In [ ]:
%pip install -qU langchain langgraph langchain-openai python-dotenv pydantic

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

print("✅ 환경 설정 완료!")

---
## 1. 평가 스키마 정의

`with_structured_output`으로 평가 결과를 구조화합니다.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 평가 결과 스키마
class Feedback(BaseModel):
    grade: Literal["funny", "not funny"] = Field(
        description="농담이 재미있는지 평가"
    )
    feedback: str = Field(
        description="재미없다면 개선 방안 제시"
    )

# LLM 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 구조화된 출력 evaluator
evaluator = llm.with_structured_output(Feedback)

print("✅ Evaluator 설정 완료!")

---
## 2. State 정의

In [ ]:
from typing import TypedDict

class State(TypedDict):
    joke: str           # 생성된 농담
    topic: str          # 농담 주제
    feedback: str       # 평가자 피드백
    funny_or_not: str   # 평가 결과
    iteration: int      # 반복 횟수

print("✅ State 정의 완료!")

---
## 3. Generator와 Evaluator 노드

In [ ]:
def llm_call_generator(state: State):
    """Generator: 농담 생성"""
    iteration = state.get("iteration", 0) + 1
    print(f"\n✍️ Generator: 농담 생성 (시도 #{iteration})")
    
    # 피드백이 있으면 반영하여 재생성
    if state.get("feedback"):
        prompt = f"{state['topic']}에 대한 농담을 써줘. 이전 피드백을 반영해: {state['feedback']}"
    else:
        prompt = f"{state['topic']}에 대한 재미있는 농담을 써줘"
    
    result = llm.invoke(prompt)
    print(f"   📝 생성된 농담: {result.content[:80]}...")
    
    return {"joke": result.content, "iteration": iteration}

def llm_call_evaluator(state: State):
    """Evaluator: 농담 평가"""
    print(f"\n🔍 Evaluator: 농담 평가 중...")
    
    grade = evaluator.invoke(f"다음 농담을 평가해줘: {state['joke']}")
    
    print(f"   {'✅' if grade.grade == 'funny' else '❌'} 결과: {grade.grade}")
    if grade.grade != "funny":
        print(f"   💬 피드백: {grade.feedback}")
    
    return {"funny_or_not": grade.grade, "feedback": grade.feedback}

print("✅ Generator/Evaluator 노드 정의 완료!")

---
## 4. 조건부 라우팅 함수

In [ ]:
def route_joke(state: State):
    """평가 결과에 따라 종료 또는 재시도"""
    # 최대 반복 횟수 제한
    if state.get("iteration", 0) >= 3:
        print("\n⚠️ 최대 시도 횟수 도달")
        return "Accepted"
    
    if state["funny_or_not"] == "funny":
        print("\n🎉 재미있는 농담 완성!")
        return "Accepted"
    else:
        print("\n🔄 피드백 반영하여 재시도")
        return "Rejected + Feedback"

print("✅ 라우팅 함수 정의 완료!")

---
## 5. 그래프 구성

In [ ]:
from langgraph.graph import StateGraph, START, END

# 워크플로우 구성
builder = StateGraph(State)

# 노드 추가
builder.add_node("llm_call_generator", llm_call_generator)
builder.add_node("llm_call_evaluator", llm_call_evaluator)

# 엣지 추가
builder.add_edge(START, "llm_call_generator")
builder.add_edge("llm_call_generator", "llm_call_evaluator")

# 조건부 엣지: 평가 결과에 따라 종료 또는 재시도
builder.add_conditional_edges(
    "llm_call_evaluator",
    route_joke,
    {
        "Accepted": END,
        "Rejected + Feedback": "llm_call_generator"
    }
)

# 컴파일
optimizer_workflow = builder.compile()
print("✅ Evaluator-Optimizer 워크플로우 컴파일 완료!")

In [ ]:
from IPython.display import Image, display
try:
    display(Image(optimizer_workflow.get_graph().draw_mermaid_png()))
except:
    print("그래프: START → generator → evaluator → [END/generator]")

---
## 6. 실행 및 테스트 (스트리밍)

In [ ]:
# 스트리밍으로 반복 과정 확인
topic = "고양이"

print(f"🐱 주제: {topic}")
print("=" * 60)

final_state = None
for chunk in optimizer_workflow.stream({"topic": topic}, stream_mode="updates"):
    for node_name, updates in chunk.items():
        if "joke" in updates:
            final_state = updates

if final_state:
    print("\n" + "=" * 60)
    print("📌 최종 농담:")
    print(final_state.get("joke", ""))

In [ ]:
# 다른 주제로 테스트
result = optimizer_workflow.invoke({"topic": "프로그래머"})
print(f"\n🤖 주제: 프로그래머")
print(f"📝 농담: {result['joke']}")
print(f"🔢 시도 횟수: {result['iteration']}")

---
## 💡 정리

### Evaluator-Optimizer 핵심 개념

| 구성요소 | 역할 | 구현 방법 |
|---------|------|----------|
| **Generator** | 콘텐츠 생성 | LLM 호출 |
| **Evaluator** | 품질 평가 | `with_structured_output` |
| **Loop** | 반복 제어 | `add_conditional_edges` |

### 핵심 코드
```python
# 구조화된 평가
class Feedback(BaseModel):
    grade: Literal["pass", "fail"]
    feedback: str

evaluator = llm.with_structured_output(Feedback)

# 조건부 루프
builder.add_conditional_edges(
    "evaluator",
    route_decision,
    {"pass": END, "fail": "generator"}
)
```

👉 **다음**: CH02._07. Routing으로 AI 시스템 효율 끌어올리기